# Model testing

This notebook implements testing of BERTopic models.

To use this notebook, set a run name and a set of parameters, then run each cell sequently.
You can adjust the number of run by adding more seeds in *param_dic*.

## Hyperparameters setting

Setup parameters (for more information on parameters, see /../docs/BERTopic parameters.md) & current run name for saving purpose

In [ ]:
run_name = "_"

param_dic = {
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "n_neighbors": 15,
    "n_components": 10,
    "min_dist": 0.0,
    "random_state": [0, 1, 15, 11, 37],
    "min_cluster_size": 20,
    "min_df": 3,
    "max_df": 1.0,
    "ngram_range": (1, 3),
    "top_n_words": 5
}

## Data Loading and Preprocessing

Import data directly from the experiment datafiles.
The data consist of 1000 documents (advices) from our participants.

In [ ]:
import os
import pandas as pd

df = pd.read_csv("../results/NLP_data_advice_fulltext.csv")
docs = list(df["text"])
docs = [doc.replace('\xa0', '') for doc in docs]
classes = list(df["gen"])
id = list(df["ID"])

docs[:5]

## Embedding pre-loading

In [ ]:
from sentence_transformers import SentenceTransformer

# Pre-calculate embeddings
embedding_model = SentenceTransformer(param_dic["embedding_model"], use_auth_token=False)
embeddings = embedding_model.encode(docs, show_progress_bar=True)

## Topic fine-tuning models

In [ ]:
from bertopic.representation import KeyBERTInspired
from bertopic.representation import MaximalMarginalRelevance

# The main representation of a topic
main_representation = KeyBERTInspired()

# Additional ways of representing a topic
aspect_model2 = [KeyBERTInspired(top_n_words=20), MaximalMarginalRelevance(diversity=.5)]

# Add all models together to be run in a single `fit`
representation_model = {
   "KeyBERT": main_representation,
   "MMR":  aspect_model2 
}

## Model run

Train model for each seeds and save data (i.e., topic modeling metrics).

Metrics used (see /evaluation.py for more info about implementation):
 - Topic Coherence (C_v & C_npmi): Measures semantic similarity of words in a topic. Higher is better.
 - Topic Diversity: Uniqueness of top words across topics. Higher indicates more distinct topics.
 - Silhouette Score: Measures how similar an object is to its own cluster compared to other clusters.
 - Similarity: Measure internal topic similarity. Higher indicates more similar documents in the same topic.

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from BERTopic_evaluation_function import evaluate_model

# Data saving

data = []

# Setup different models

cluster_model = HDBSCAN(min_cluster_size=param_dic["min_cluster_size"], metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = CountVectorizer(stop_words="english", min_df=param_dic["min_df"], max_df=param_dic["max_df"], ngram_range=param_dic["ngram_range"])

for seed in param_dic["random_state"]:

    umap_model = UMAP(n_neighbors=param_dic["n_neighbors"], n_components=param_dic["n_components"], min_dist=param_dic["min_dist"], metric='cosine', random_state=seed)

    topic_model = BERTopic(

        # Pipeline models
        embedding_model=embedding_model,
        umap_model=umap_model,
        hdbscan_model=cluster_model,
        vectorizer_model=vectorizer_model,
        representation_model=representation_model,

        # Hyperparameters
        top_n_words=param_dic["top_n_words"],
        n_gram_range=param_dic["ngram_range"],
        min_topic_size="auto", #use HDBSCAN
        verbose=True,

        # General parameters
        calculate_probabilities=True,
        language="english"
    )

    topics, probs = topic_model.fit_transform(docs, embeddings)

    eval_res = evaluate_model(topic_model, docs, topics, embeddings, topk=param_dic["top_n_words"])
    data.append([seed, max(topic_model.topics_)] + [i for i in eval_res])

data = pd.DataFrame(data, columns=["seed", "nr_topic", "c_v", "c_npmi", "t_D", "silhouette", "similarity"])
data

## Saving

In [ ]:
# save BERTopic model object
embedding_model = "all-MiniLM-L6-v2"
topic_model.save("../../results/fine_tuning" + run_name, serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

In [ ]:
# save run data into .csv
data_run = [[run_name, param_dic, data["nr_topic"].mean(), data["c_v"].mean(), data["c_npmi"].mean(), data["t_D"].mean(), data["silhouette"].mean(), data["similarity"].mean()]]

df_run = pd.DataFrame(data_run, columns=["run_name", "params", "nr_topic", "c_v", "c_npmi", "diversity", "silhouette", "similarity"])

df_run.to_csv("../../results/fine_tuningresults.csv", mode="a", header=False, index=False)